In [ ]:
# Step 1: IMPORTS AND SETUP

import os
import glob
import numpy as np
import pandas as pd
import pickle
import time

import tensorflow as tf

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt



# Step 2: CONFIGURATION -

OUT_DIR = "/content/drive/MyDrive/relabelled_dataset_new/Model/Model_Hybrid_CNN_1"
NEW_DATA_DIR = "/content/drive/MyDrive/relabelled_test_dataset_new"

COLS_AXES    = ["ax", "ay", "az"]
FS           = 2720.0
WIN          = 128
HOP          = 64
F0, DELTA    = 95.0, 25.0


FEATURES_KEEP = [
    "jerk_rms", "centroid_norm", "zcr", "crest_factor_amp", "ber_f0",
    "high_ratio", "bandpower_over_entropy_amp"
]


# Step 3: RELOAD FEATURE EXTRACTION FUNCTIONS

def read_xyz_csv(path, cols=COLS_AXES):
    df = pd.read_csv(path)
    if not all(c in df.columns for c in cols):
        num_cols = [c for c in df.columns if np.issubdtype(df[c].dtype, np.number)]
        arr = df[num_cols[:3]].to_numpy(dtype=np.float32)
    else:
        arr = df[cols].to_numpy(dtype=np.float32)
    return arr
def sliding_starts(T, win, hop):
    i=0; idx=[]
    while i+win <= T:
        idx.append(i); i += hop
    return idx
def goertzel_pow(x, f, fs):
    N = len(x); k = int(0.5 + (N*f)/fs)
    w = (2*np.pi/N)*k; cw, sw = np.cos(w), np.sin(w)
    s1=0.0; s2=0.0
    for n in range(N):
        s0 = x[n] + 2*cw*s1 - s2
        s2, s1 = s1, s0
    real = s1 - s2*cw; imag = s2*sw
    return (real*real + imag*imag)/N
def spectral_entropy(x, fs):
    P = np.abs(np.fft.rfft(x - np.mean(x)))**2
    P /= (np.sum(P) + 1e-12)
    return float(-np.sum(P * np.log(P + 1e-12)) / np.log(len(P)))
def zero_crossing_rate(x):
    return float(np.mean(np.abs(np.diff(np.signbit(x - np.mean(x))))))
def band_energy_ratio(x, fs, f_lo, f_hi):
    freqs = np.fft.rfftfreq(len(x), 1/fs)
    X = np.abs(np.fft.rfft(x - np.mean(x)))**2
    total = np.sum(X) + 1e-12
    return float(np.sum(X[(freqs>=f_lo) & (freqs<=f_hi)]) / total)
def spectral_stats_basic(x, fs, f0=F0):
    freqs = np.fft.rfftfreq(len(x), 1/fs)
    S = np.abs(np.fft.rfft(x - x.mean())) + 1e-9
    centroid_norm= float(np.sum(freqs*S) / np.sum(S))/(fs/2.0)
    high = freqs>=2.5*f0
    high_ratio = float((np.sum(S[high])+1e-9) / (np.sum(S[(freqs>=f0) & (freqs<=3*f0)])+1e-9))
    return centroid_norm, high_ratio
def crest_factor(x):
    rms = np.sqrt(np.mean(x*x) + 1e-12)
    return float(np.max(np.abs(x)) / (rms + 1e-12))
def compute_selected_features(ax, ay, az, fs=FS):
    a = np.sqrt(ax*ax + ay*ay + az*az).astype(np.float32)
    cen_n, hr = spectral_stats_basic(a, fs)
    jerk = np.diff(a, prepend=a[0]) * fs
    jerk_rms = float(np.sqrt(np.mean(jerk*jerk)))
    zcr = zero_crossing_rate(a)
    ber_f0 = band_energy_ratio(a, fs, F0-10, F0+10)
    band_pow = band_energy_ratio(a, fs, F0-2*DELTA, F0+2*DELTA) * (np.sum(np.abs(np.fft.rfft(a - np.mean(a)))**2)+1e-12)
    Hn = spectral_entropy(a, fs)
    bandpower_over_entropy_amp = band_pow / (Hn + 1e-12)
    cf = crest_factor(a)
    return { "jerk_rms": jerk_rms, "centroid_norm": cen_n, "zcr": zcr, "crest_factor_amp": cf,
        "ber_f0": ber_f0, "high_ratio": hr, "bandpower_over_entropy_amp": bandpower_over_entropy_amp }


# Step 4: LOAD ARTIFACTS AND PROCESS NEW DATA

print("--- Loading Pre-processing Objects ---")

with open(os.path.join(OUT_DIR, "scaler.pkl"), "rb") as f:
    scaler = pickle.load(f)
with open(os.path.join(OUT_DIR, "label_encoder.pkl"), "rb") as f:
    le = pickle.load(f)
print("Scaler and Label Encoder loaded successfully.")

print("\n--- Processing New Offline Test Data ---")
all_windows, all_features, all_labels = [], [], []
class_dirs = sorted([d for d in glob.glob(os.path.join(NEW_DATA_DIR, "*")) if os.path.isdir(d)])
assert class_dirs, f"No class folders found under {NEW_DATA_DIR}"

for cdir in class_dirs:
    label = os.path.basename(cdir)
    for fp in sorted(glob.glob(os.path.join(cdir, "*.csv"))):
        xyz = read_xyz_csv(fp)
        for s in sliding_starts(len(xyz), WIN, HOP):
            seg = xyz[s:s+WIN, :]; all_windows.append(seg)
            feats = compute_selected_features(seg[:,0], seg[:,1], seg[:,2], fs=FS)
            all_features.append([feats[k] for k in FEATURES_KEEP]); all_labels.append(label)

X_windows_new = np.asarray(all_windows, dtype=np.float32)
X_features_new = np.asarray(all_features, dtype=np.float32)
y_true_labels = le.transform(np.array(all_labels, dtype=object))
X_features_new_s = scaler.transform(X_features_new)

print(f"\nNew data processed. Total test windows: {len(y_true_labels)}")


# Step 5: TEST THE .KERAS MODEL

print("\n--- Testing Keras Model from New Search ---")
keras_model = tf.keras.models.load_model(os.path.join(OUT_DIR, "best_hybrid_model.keras"))
print("Keras model loaded.")

y_pred_keras_probs = keras_model.predict([X_windows_new, X_features_new_s])
y_pred_keras = np.argmax(y_pred_keras_probs, axis=1)

keras_accuracy = accuracy_score(y_true_labels, y_pred_keras)
print(f"Keras Model Accuracy: {keras_accuracy:.4f}\n")
print("Keras Classification Report:")
print(classification_report(y_true_labels, y_pred_keras, target_names=le.classes_))


# Step 6: TEST THE .TFLITE INT8 MODEL

print("\n--- Testing TFLite INT8 Model from New Search ---")
interpreter = tf.lite.Interpreter(model_path=os.path.join(OUT_DIR, "model_int8.tflite"))
interpreter.allocate_tensors()
print("TFLite model loaded.")

input_details = interpreter.get_input_details(); output_details = interpreter.get_output_details()
y_pred_tflite = []

for i in range(len(X_windows_new)):
    window_sample = X_windows_new[i:i+1]
    feature_sample_s = X_features_new_s[i:i+1]

    cnn_input_scale, cnn_input_zero_point = input_details[0]['quantization']
    window_sample_q = (window_sample / cnn_input_scale + cnn_input_zero_point).astype(np.int8)
    feature_input_scale, feature_input_zero_point = input_details[1]['quantization']
    feature_sample_s_q = (feature_sample_s / feature_input_scale + feature_input_zero_point).astype(np.int8)

    interpreter.set_tensor(input_details[0]['index'], window_sample_q)
    interpreter.set_tensor(input_details[1]['index'], feature_sample_s_q)
    interpreter.invoke()

    output_data_q = interpreter.get_tensor(output_details[0]['index'])
    output_scale, output_zero_point = output_details[0]['quantization']
    output_data = (output_data_q.astype(np.float32) - output_zero_point) * output_scale
    y_pred_tflite.append(np.argmax(output_data))

tflite_accuracy = accuracy_score(y_true_labels, y_pred_tflite)
print(f"Final TFLite INT8 Model Accuracy: {tflite_accuracy:.4f}\n")
print("TFLite Classification Report:")
print(classification_report(y_true_labels, y_pred_tflite, target_names=le.classes_))

# Step 7: VISUAL COMPARISON (CONFUSION MATRIX)

print("\n--- Generating Final Confusion Matrices on Offline Data ---")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('New Tuner Model Performance on New Offline Test Data', fontsize=16)

cm_keras = confusion_matrix(y_true_labels, y_pred_keras)
sns.heatmap(cm_keras, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title(f'Keras Model (Acc: {keras_accuracy:.3f})')
axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')

cm_tflite = confusion_matrix(y_true_labels, y_pred_tflite)
sns.heatmap(cm_tflite, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[1])
axes[1].set_title(f'TFLite INT8 Model (Acc: {tflite_accuracy:.3f})')
axes[1].set_ylabel('True Label'); axes[1].set_xlabel('Predicted Label')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()